# Fontbelle - consolidation CTD + Aqua TROLL + centrale OTT

## 1. Imports

In [ ]:
import os
import re
import unicodedata
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Fontbelle\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes\CTD")
VUSITU_PATH = os.path.join(BASE, r"Données brutes\TROLL")
OTT_PATH    = os.path.join(BASE, r"Données brutes\OTT")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"

OLDDATA_PATH   = os.path.join(BASE, "Fontbelle__consolide_OLD.xlsx")
UTC_CTD_PATH   = os.path.join(BASE, "UTC_CTD.xlsx")
UTC_TROLL_PATH = os.path.join(BASE, "UTC_Troll.xlsx")

PUNCTUAL_NIVEAU  = os.path.join(BASE, "punctual_measurements_Niveau.xlsx")
PUNCTUAL_CONDUCT = os.path.join(BASE, "punctual_measurements_Conducti.xlsx")
PLUIE_PATH = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Saint Sauveur\Gaetan\Données brutes\Pluie_BV_Ouysse.csv"

SORTIE_CONSOLIDE = os.path.join(BASE, "Fontbelle_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, "Fontbelle_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, "Graphes.svg")

PREFIXE_CTD  = "Fontbelle"
BARO_COL     = "Patm Ouysse Calès [hPa]"
PAS          = "1h"
HPA_EN_CMH2O = 1.019716

## 3. Fonctions de lecture

In [ ]:
def _sans_accents(t):
    d = unicodedata.normalize("NFKD", str(t))
    return "".join(c for c in d if not unicodedata.combining(c)).lower()


def _lire_lignes(chemin, encodages=("utf-8-sig", "utf-8", "cp1252", "latin1")):
    """Essaie plusieurs encodages au lieu de figer latin1."""
    for enc in encodages:
        try:
            with open(chemin, "r", encoding=enc) as f:
                return f.read().splitlines(), enc
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Aucun encodage ne convient pour {chemin}")


def _en_datetime(serie, dayfirst=True):
    """Essaie les formats connus, garde celui qui convertit le plus de lignes."""
    txt = serie.astype("string").str.strip()
    meilleur, n_ok = None, -1
    for fmt in ("%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M"):
        e = pd.to_datetime(txt, format=fmt, errors="coerce")
        if e.notna().sum() > n_ok:
            meilleur, n_ok = e, e.notna().sum()
    if n_ok < len(txt):
        s = pd.to_datetime(txt, errors="coerce", dayfirst=dayfirst)
        if s.notna().sum() > n_ok:
            meilleur = s
    return meilleur


def fichiers_correspondant(path, motif):
    """Fichiers d'un dossier contenant `motif`, triés par numéro."""
    noms = [f for f in os.listdir(path)
            if motif.lower() in f.lower() and f.lower().endswith((".csv", ".txt", ".mon"))]
    return sorted(noms, key=lambda n: (int(re.findall(r"\d+", n)[0]) if re.findall(r"\d+", n) else 10**9, n))


def lire_fichier_CTD(nom_fichier, path=CTD_PATH):
    """Export Diver. En-tête cherchée (fin de skiprows=63), pied reconnu (fin de iloc[:-1])."""
    chemin = os.path.join(path, nom_fichier)
    lignes, encodage = _lire_lignes(chemin)

    entete = next((i for i, l in enumerate(lignes[:200])
                   if _sans_accents(l).lstrip("\ufeff").startswith("date/time")), None)
    if entete is None:
        raise ValueError(f"En-tête « Date/time » introuvable dans {nom_fichier}")

    df = pd.read_csv(chemin, sep=";", encoding=encodage, skiprows=entete,
                     header=0, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    col_date = df.columns[0]

    brut = df[col_date].astype("string")
    df = df.loc[~(brut.map(lambda v: pd.notna(v) and "end of data" in _sans_accents(v)).fillna(False)
                  | brut.isna() | (brut.str.strip() == ""))].copy()

    df["Date/time"] = _en_datetime(df[col_date]).dt.round(PAS)
    for col in df.columns:
        if col not in ("Date/time", col_date):
            df[col] = pd.to_numeric(df[col].astype("string").str.strip()
                                    .str.replace(",", ".", regex=False), errors="coerce")

    # Conductivité : le facteur vient de l'UNITÉ entre crochets, pas du libellé exact.
    for col in list(df.columns):
        if "cond" in _sans_accents(col):
            u = re.search(r"\[([^\]]*)\]", col)
            u = _sans_accents(u.group(1)) if u else ""
            df[col] = df[col] * (1000.0 if u.startswith("ms/cm") else 1.0)
            df = df.rename(columns={col: "Cond_(µS/cm)"})
            break

    return df.loc[df["Date/time"].notna()].sort_values("Date/time").reset_index(drop=True)


#: Libellés VuSitu, numéro de série retiré → vos noms de colonnes.
NOMS_TROLL = {
    "conductivité spécifique (µs/cm)": "Cond_Troll_(µS/cm)",
    "température (°c)":                "température_Troll_(°C)",
    "turbidité (ntu)":                 "Turbidity_Troll_(NTU)",
    "concentration rdo (mg/l)":        "O2_Troll_(mg/l)",
    "saturation rdo (%sat)":           "O2 (%Sat)",
    "fluorescence de chlorophylle-a (rfu)": "FluorescenceChloro_a_Troll_(RFU)",
    "concentration de chlorophylle-a (µg/l)": "ConcentrationChloro_a_(µg/l)",
}


def renommer_colonnes(df):
    """Retire le n° de série du capteur, puis applique vos noms.

    Remplace la table de 40 correspondances écrites à la main : le numéro
    entre parenthèses en fin de libellé (« ... (737318) ») part par regex,
    donc un changement de sonde ne demande plus rien.
    """
    renommage = {}
    for col in df.columns:
        cle = re.sub(r"\s*\(\d{4,}\)\s*$", "", str(col)).strip().lower().replace("μ", "µ")
        if cle in NOMS_TROLL:
            renommage[col] = NOMS_TROLL[cle]
    return df.rename(columns=renommage)


def lire_fichier_vusitu(nom_fichier, path=VUSITU_PATH):
    """Export VuSitu (Aqua TROLL) : guillemets retirés, colonnes normalisées."""
    lignes, _ = _lire_lignes(os.path.join(path, nom_fichier))
    df = pd.read_csv(StringIO("\n".join(l.replace('"', "") for l in lignes)), sep=",")
    col_date = next((c for c in df.columns if "date" in _sans_accents(c)), df.columns[0])
    df["DATE"] = _en_datetime(df[col_date]).dt.round(PAS)
    return renommer_colonnes(df.drop(columns=[col_date], errors="ignore"))


#: Gamme physique de chaque type de capteur. La clé est un mot-clé cherché
#: dans le nom de colonne, donc une nouvelle voie est couverte sans rien ajouter.
GAMMES = {"cond": (200, 5000), "temp": (-2, 30), "niveau": (0, 3000),
          "turbid": (0, 4000), "o2": (0, 25), "chloro": (0, 500)}


def appliquer_gammes(df, gammes=GAMMES):
    """Écarte de chaque voie ce qui est physiquement impossible.

    À appliquer avant toute mise en priorité : une voie muette qui renvoie 0
    (la voie C2 de la centrale le fait 2 425 fois) passerait sinon le
    contrôle une fois recalée sur le capteur prioritaire.
    """
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        for cle, (mini, maxi) in gammes.items():
            if cle in _sans_accents(col):
                hors = (df[col] < mini) | (df[col] > maxi)
                if hors.any():
                    print(f"  {col:38s} {int(hors.sum()):6d} hors gamme [{mini}, {maxi}]")
                    df.loc[hors, col] = np.nan
                break
    return df


#: Colonnes de la centrale OTT → noms du projet.
NOMS_OTT = {"level": "Niveau_OTT_(cm)",
            "c1": "Cond_CTDOTT_(µS/cm)",   "t1": "Temp_CTDOTT_(°C)",      # voie CTD
            "c2": "Cond_TrollOTT_(µS/cm)", "t2": "Temp_TrollOTT_(°C)",    # voie TROLL
            "turbi": "Turbidity_TrollOTT_(NTU)", "o2": "O2_TrollOTT_(mg/l)",
            "chlorophyl": "FluorescenceChloro_a_TrollOTT_(RFU)"}


def lire_fichier_OTT(nom_fichier, path=OTT_PATH):
    """Export de la centrale d'acquisition (déjà en UTC).

    Deux pièges de ce format, traités ici : la valeur **-99999** qui code
    l'absence de mesure (sans quoi la turbidité moyenne vaut −42 249 NTU)
    et les horodatages en double. Les valeurs physiquement impossibles des
    voies débranchées sont écartées plus loin par `appliquer_gammes`.
    """
    chemin = os.path.join(path, nom_fichier)
    _, encodage = _lire_lignes(chemin)
    df = pd.read_csv(chemin, sep=";", encoding=encodage, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]

    col_date = df.columns[0]
    df["DATE"] = _en_datetime(df[col_date])
    df = df.drop(columns=[col_date])

    for col in df.columns:
        if col == "DATE":
            continue
        df[col] = pd.to_numeric(df[col].astype("string").str.strip()
                                .str.replace(",", ".", regex=False), errors="coerce")
        df.loc[df[col].isin([-99999, -9999, 9999]), col] = np.nan

    df = df.rename(columns={c: NOMS_OTT[c.lower()] for c in df.columns if c.lower() in NOMS_OTT})
    df = df.dropna(subset=["DATE"]).groupby("DATE", as_index=False).median(numeric_only=True)
    return df.sort_values("DATE").reset_index(drop=True)


def convertir_en_utc0(df, nom_fichier, metadata, col_date="Date/time",
                      col_nom="Nom fichier", col_utc="UTC fichier"):
    """Ramène les horodatages en UTC d'après la table des campagnes.

    Accepte « UTC+1 », « UTC », 1, 1.0..., au lieu de ne reconnaître que deux
    chaînes exactes et de renvoyer 0 en silence pour tout le reste.
    """
    ligne = metadata.loc[metadata[col_nom] == nom_fichier, col_utc]
    if ligne.empty:
        raise ValueError(f"Fuseau horaire non trouvé pour : {nom_fichier}")
    v = ligne.values[0]
    if isinstance(v, (int, float)) and pd.notna(v):
        decalage = float(v)
    else:
        m = re.search(r"([+-]?\d+(?:[.,]\d+)?)", str(v))
        decalage = float(m.group(1).replace(",", ".")) if m else 0.0

    df = df.copy()
    df[col_date] = df[col_date] - pd.Timedelta(hours=decalage)
    return df, decalage


def graphe(traces, titre="", ylab="", points=None, col_point=None, sortie_html=None):
    """Utilitaire Plotly. `traces` = liste de (série, nom, couleur).

    Utilise Scattergl (rendu WebGL) : une chronique horaire pluriannuelle
    s'affiche sans saturer le navigateur, contrairement à Scatter.
    """
    fig = go.Figure()
    for serie, nom, couleur in traces:
        fig.add_trace(go.Scattergl(x=serie.index, y=serie, mode="lines", name=nom,
                                   line=dict(color=couleur, width=1.3)))
    if points is not None and col_point in points.columns:
        corr = points["Correction"] if "Correction" in points.columns else pd.Series("Non", index=points.index)
        fig.add_trace(go.Scattergl(
            x=points["Datetime"], y=points[col_point], mode="markers", name="points de contrôle",
            marker=dict(color=["red" if str(v).strip() == "Oui" else "royalblue" for v in corr],
                        symbol="x", size=10)))
    fig.update_layout(title=titre, xaxis_title="Date", yaxis_title=ylab,
                      template="plotly_white", hovermode="x unified")
    if sortie_html:
        fig.write_html(sortie_html)
        print(f"Graphique sauvegardé : {sortie_html}")
    fig.show()        

## 4. Lecture CTD : conversion UTC et compensation barométrique

La conversion UTC est appliquée à chaque campagne d'après `UTC_CTD.xlsx` (fonction
`convertir_en_utc0` ci-dessus) ; le décalage retenu par fichier est affiché.

In [ ]:
metadata = pd.read_excel(UTC_CTD_PATH)
baro_data = pd.read_excel(BARO_PATH)
baro_data["DATE"] = pd.to_datetime(baro_data["DATE"], errors="coerce")
baro_data = baro_data[["DATE", BARO_COL]].dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux, echecs = [], []
for nom_fichier in fichiers_correspondant(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = convertir_en_utc0(lire_fichier_CTD(nom_fichier), nom_fichier, metadata)
        print(f"  {nom_fichier:45s} UTC+{decalage:g} → UTC   ({len(CTD)} lignes)")
        merged_df = pd.merge(CTD, baro_data, left_on="Date/time", right_on="DATE", how="left")

        # ── Les deux pressions dans la MÊME unité avant de soustraire ──
        merged_df["Niveau_(cm)"] = merged_df["Pression[cmH2O]"] - merged_df[BARO_COL] * HPA_EN_CMH2O

        merged_df = merged_df.rename(columns={"Température[°C]": "Temp_(°C)"})
        morceaux.append(merged_df[["Date/time", "Niveau_(cm)", "Pression[cmH2O]",
                                   BARO_COL, "Cond_(µS/cm)", "Temp_(°C)"]])
    except Exception as e:
        echecs.append((nom_fichier, str(e)))

for nom, err in echecs:
    print(f"⚠ {nom} : {err}")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s) CTD, {len(merge_ctd_df)} enregistrements en UTC.")
merge_ctd_df.head()

## 5. Raccordement à l'ancienne chronique

Le décalage était estimé sur **un seul couple de valeurs** (`iloc[-1]` et `iloc[0]`), qui
portent tout le bruit de mesure. On prend la médiane des différences sur la fenêtre commune,
avec son incertitude. Votre graphe de contrôle des 7 jours autour de la jonction est conservé.

In [ ]:
def estimer_decalage(serie_ref, serie_a_caler, fenetre_j=7, min_points=3):
    """Constante à ajouter à `serie_a_caler` pour rejoindre `serie_ref`."""
    a, n = serie_ref.dropna().sort_index(), serie_a_caler.dropna().sort_index()
    if a.empty or n.empty:
        return 0.0, np.nan, 0, "série vide"
    commun = a.index.intersection(n.index)
    if len(commun) >= min_points:
        ecarts, methode = (a.loc[commun] - n.loc[commun]).to_numpy(), "recouvrement"
    else:
        fin = a[a.index >= a.index.max() - pd.Timedelta(days=fenetre_j)]
        debut = n[n.index <= n.index.min() + pd.Timedelta(days=fenetre_j)]
        if len(fin) < min_points or len(debut) < min_points:
            return 0.0, np.nan, 0, "pas de recouvrement exploitable"
        ecarts = np.array([np.median(fin.to_numpy()) - np.median(debut.to_numpy())])
        methode = "extrapolation (à vérifier)"
    d = float(np.median(ecarts))
    u = float(1.4826 * np.median(np.abs(ecarts - d))) if ecarts.size > 1 else np.nan
    return d, u, int(ecarts.size), methode


olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df["DATE"] = pd.to_datetime(olddata_df["DATE"], errors="coerce")
olddata_df = olddata_df.dropna(subset=["DATE"]).sort_values("DATE")

last_date_old, first_date_new = olddata_df["DATE"].max(), merge_ctd_df["DATE"].min()
print(f"Dernière date ancienne : {last_date_old}")
print(f"Première date nouvelle : {first_date_new}\n")

ancien_i, nouveau_i = olddata_df.set_index("DATE"), merge_ctd_df.set_index("DATE")
d_niv, u_niv, n_niv, m_niv = estimer_decalage(ancien_i["Niveau_(cm)"], nouveau_i["Niveau_(cm)"])
d_cnd, u_cnd, n_cnd, m_cnd = estimer_decalage(ancien_i["Cond_CTD_(µS/cm)"], nouveau_i["Cond_(µS/cm)"])
print(f"Niveau       : décalage {d_niv:+8.2f} cm    (± {u_niv:.2f}, n={n_niv}, {m_niv})")
print(f"Conductivité : décalage {d_cnd:+8.2f} µS/cm (± {u_cnd:.2f}, n={n_cnd}, {m_cnd})")

merge_ctd_df["Niveau_corr"] = merge_ctd_df["Niveau_(cm)"] + d_niv
merge_ctd_df["Conducti_corr"] = merge_ctd_df["Cond_(µS/cm)"] + d_cnd

# ── Graphe de contrôle : 7 jours autour de la jonction ──────────────────────
old_f = olddata_df[olddata_df["DATE"] >= last_date_old - pd.Timedelta(days=7)]
new_f = merge_ctd_df[merge_ctd_df["DATE"] <= first_date_new + pd.Timedelta(days=7)]

fig, ax1 = plt.subplots(figsize=(11, 3))
ax1.plot(old_f["DATE"], old_f["Niveau_(cm)"], label="Niveau (consolidé)", color="green")
ax1.plot(new_f["DATE"], new_f["Niveau_corr"], label="Niveau (nouveau, corrigé)", color="blue")
ax1.set_ylabel("Niveau (cm)")
ax2 = ax1.twinx()
ax2.plot(old_f["DATE"], old_f["Cond_CTD_(µS/cm)"], label="Conductivité (consolidé)", color="orange")
ax2.plot(new_f["DATE"], new_f["Conducti_corr"], label="Conductivité (nouveau, corrigé)", color="red")
ax2.set_ylabel("Conductivité (µS/cm)")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=2)
plt.tight_layout()
plt.show()

## 6. Lecture des données Aqua TROLL (VuSitu)

In [ ]:
metadata_troll

In [ ]:
print(metadata_troll.columns.tolist())

In [ ]:
metadata_troll = pd.read_excel(UTC_TROLL_PATH, sheet_name=0)

morceaux = []
for nom_fichier in fichiers_correspondant(VUSITU_PATH, "VuSitu"):
    try:
        df_v, decalage = convertir_en_utc0(lire_fichier_vusitu(nom_fichier), nom_fichier,
                                           metadata_troll, col_date="DATE", col_utc="Fuseau détecté")
        print(f"  {nom_fichier:45s} UTC+{decalage:g} → UTC   ({len(df_v)} lignes)")
        morceaux.append(df_v)
    except Exception as e:
        print(f"⚠ {nom_fichier} : {e}")

merge_troll_df = pd.concat(morceaux, ignore_index=True).dropna(subset=["DATE"]).sort_values("DATE")
print(f"\n{len(morceaux)} fichier(s) TROLL, {len(merge_troll_df)} enregistrements en UTC.")
merge_troll_df.head()

## 7. Lecture de la centrale OTT

Nouvelle source. Elle porte, en doublon des exports directs, la voie **CTD** (`C1`, `T1`),
la voie **TROLL** (`C2`, `T2`, `Turbi`, `O2`, `Chlorophyl`) et son propre **niveau** (`level`).

In [ ]:
morceaux = []
for nom_fichier in fichiers_correspondant(OTT_PATH, ""):
    print(f"  {nom_fichier}")
    morceaux.append(lire_fichier_OTT(nom_fichier))

merge_ott_df = (pd.concat(morceaux, ignore_index=True).groupby("DATE", as_index=False)
                .median(numeric_only=True).sort_values("DATE")) if morceaux else pd.DataFrame(columns=["DATE"])

if not merge_ott_df.empty:
    print(f"\n{len(merge_ott_df)} pas de temps. Couverture par voie :")
    for c in merge_ott_df.columns:
        ok = merge_ott_df[c].notna() if c != "DATE" else None
        if ok is not None and ok.any():
            print(f"  {c:38s} {ok.sum():6d} pts  "
                  f"({merge_ott_df.loc[ok, 'DATE'].min():%Y-%m-%d} → {merge_ott_df.loc[ok, 'DATE'].max():%Y-%m-%d})")
merge_ott_df.head()

## 8. Fusion des trois sources

Chaque capteur garde sa colonne. Les colonnes de travail sont construites par ordre de
priorité : le premier disponible gagne, le secours ne comble que les trous et il est
**recalé** sur le prioritaire (décalage médian) pour ne pas créer de marche.

Avant cela, `appliquer_gammes` écarte de **chaque voie** ce qui est physiquement impossible
(voie débranchée qui renvoie 0, pics à 161 664 µS/cm de la centrale, sonde hors d'eau) :
sinon ces valeurs entreraient dans la synthèse une fois recalées.

`PREFERENCES_PERIODE` permet d'imposer une voie sur une période donnée, pour un ou
plusieurs paramètres.

In [ ]:
# Ordre de priorité : conductivité et température au TROLL, niveau à la CTD.
PRIORITES = {
    "Niveau_(cm)":  ["Niveau_OTT_(cm)", "Niveau_CTD_(cm)"],
    "Conductivité": ["Cond_Troll_(µS/cm)", "Cond_TrollOTT_(µS/cm)",
                     "Cond_CTD_(µS/cm)", "Cond_CTDOTT_(µS/cm)"],
    "Température":  ["température_Troll_(°C)", "Temp_TrollOTT_(°C)",
                     "Temp _CTD(°C)", "Temp_CTDOTT_(°C)"],
    # Voies portées seulement par la TROLL, en direct puis via la centrale.
    "Turbidité_(NTU)":    ["Turbidity_Troll_(NTU)", "Turbidity_TrollOTT_(NTU)"],
    "O2_(mg/l)":          ["O2_Troll_(mg/l)", "O2_TrollOTT_(mg/l)"],
    "Chlorophylle_(RFU)": ["FluorescenceChloro_a_Troll_(RFU)",
                           "FluorescenceChloro_a_TrollOTT_(RFU)"],
}

# Voie à privilégier sur une période : (début, fin, voie, [paramètres]).
PREFERENCES_PERIODE = [
    # ("2025-06-01", "2025-08-01", "Cond_CTD_(µS/cm)", ["Conductivité"]),
]


def completer_avec_secours(df, cible, sources):
    """Premier disponible gagne ; le secours comble les trous, recalé."""
    presentes = [c for c in sources if c in df.columns and df[c].notna().any()]
    valeur = df[presentes[0]].copy() if presentes else pd.Series(np.nan, index=df.index)
    source = pd.Series(pd.NA, index=df.index, dtype="object")
    if presentes:
        source[valeur.notna()] = presentes[0]

    for nom in presentes[1:]:
        secours = df[nom]
        commun = valeur.notna() & secours.notna()
        decalage = float((valeur[commun] - secours[commun]).median()) if commun.any() else 0.0
        a_combler = valeur.isna() & secours.notna()
        valeur[a_combler] = secours[a_combler] + decalage
        source[a_combler] = nom
        print(f"    {nom:26s} secours, décalage {decalage:+8.2f}, {int(a_combler.sum()):6d} pas comblés")

    df[cible], df[f"{cible}_source"] = valeur, source
    return df


# ── Assemblage sur une grille horaire ───────────────────────────────────────
ctd = (merge_ctd_df.set_index("DATE")[["Niveau_corr", "Conducti_corr", "Temp_(°C)"]]
       .rename(columns={"Niveau_corr": "Niveau_CTD_(cm)", "Conducti_corr": "Cond_CTD_(µS/cm)",
                        "Temp_(°C)": "Temp _CTD(°C)"}))

full_data = None
for nom, df in [("ancien", olddata_df.set_index("DATE")), ("CTD", ctd),
                ("TROLL", merge_troll_df.set_index("DATE")),
                ("OTT", merge_ott_df.set_index("DATE") if not merge_ott_df.empty else pd.DataFrame())]:
    if df.empty:
        continue
    # Colonnes numériques seulement, et un enregistrement par heure : les
    # campagnes CTD se recouvrent, or concat(axis=1) refuse un index en double.
    df = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    df = df[df.index.notna()].groupby(level=0).median().sort_index()
    if full_data is None:
        full_data = df
        continue
    # Une colonne portée par deux sources est COMBINÉE (la récente l'emporte),
    # pas écartée : `~columns.duplicated()` jetait les données rejouées.
    full_data = full_data.reindex(full_data.index.union(df.index))
    for col in df.columns:
        d = df[col].reindex(full_data.index)
        full_data[col] = d.combine_first(full_data[col]) if col in full_data.columns else d

full_data = full_data.reindex(pd.date_range(full_data.index.min(), full_data.index.max(), freq=PAS))
full_data.index.name = "DATE"

# Gammes physiques : toutes les voies, toutes sources confondues, avant priorités.
print("Valeurs hors gamme écartées :")
full_data = appliquer_gammes(full_data)

for cible, sources in PRIORITES.items():
    print(f"{cible} :")
    full_data = completer_avec_secours(full_data, cible, sources)

# ── Voie privilégiée sur une période ────────────────────────────────────────
for debut, fin, voie, parametres in PREFERENCES_PERIODE:
    periode = (full_data.index >= pd.to_datetime(debut)) & (full_data.index <= pd.to_datetime(fin))
    for cible in parametres:
        hors = full_data[cible].notna() & full_data[voie].notna() & ~periode
        decalage = float((full_data.loc[hors, cible] - full_data.loc[hors, voie]).median()) if hors.any() else 0.0
        full_data.loc[periode, cible] = full_data.loc[periode, voie] + decalage
        full_data.loc[periode & full_data[voie].notna(), f"{cible}_source"] = voie + " (imposé)"
        print(f"{cible} : {voie} imposé du {debut} au {fin} (décalage {decalage:+.2f})")

full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\n Consolidé : {SORTIE_CONSOLIDE}  ({len(full_data)} pas de temps)")

## 9. Niveau - AVANT correction

In [ ]:
points_niveau = pd.read_excel(PUNCTUAL_NIVEAU)
points_niveau["Datetime"] = pd.to_datetime(points_niveau["Jour"], dayfirst=True, errors="coerce")
niveau_avant = full_data["Niveau_(cm)"].copy()

graphe([(niveau_avant, "Niveau (avant)", "grey")],
       titre="Niveau — AVANT correction", ylab="Niveau (cm)",
       points=points_niveau, col_point="Hauteur (cm)")

## 10. Niveau - correction et graphe APRÈS


In [ ]:
# Périodes écartées : (début, fin, motif). Valeur isolée = même date deux fois.
PERIODES_ECARTEES_NIVEAU = [
    ("2019-04-18 12:00", "2019-04-18 12:00", "pic isolé"),
    ("2020-07-29 14:00", "2020-07-29 15:00", "pic isolé"),
    ("2020-09-23 13:00", "2020-09-23 13:00", "pic isolé"),
    ]


def ecarter_periodes(serie, periodes):
    """Passe à NaN les périodes listées et retourne l'historique."""
    serie, journal = serie.copy(), []
    for debut, fin, motif in periodes:
        debut, fin = sorted([pd.to_datetime(debut), pd.to_datetime(fin)])
        m = (serie.index >= debut) & (serie.index <= fin)
        journal.append({"début": debut, "fin": fin, "n écartés": int((m & serie.notna()).sum()),
                        "motif": motif})
        serie[m] = np.nan
    return serie, pd.DataFrame(journal)


def caler_sur_points_de_controle(serie, points, col_valeur, tolerance_h=1):
    """Recale la série sur les mesures ponctuelles marquées « Oui »."""
    serie, journal = serie.copy(), []
    for _, ligne in points.dropna(subset=["Datetime"]).sort_values("Datetime").iterrows():
        date, valeur = ligne["Datetime"], ligne.get(col_valeur)
        if pd.isna(valeur) or str(ligne.get("Correction", "Non")).strip() != "Oui":
            continue
        mesures = serie.dropna()
        if mesures.empty:
            continue
        i = mesures.index[np.abs((mesures.index - date).to_numpy()).argmin()]
        if abs((i - date).total_seconds()) > tolerance_h * 3600:
            journal.append({"date": date, "terrain": valeur, "décalage": np.nan,
                            "note": "pas de mesure à moins d'1 h"})
            continue
        decalage = float(valeur - serie.loc[i])
        serie.loc[serie.index >= date] += decalage
        journal.append({"date": date, "terrain": valeur, "décalage": round(decalage, 2),
                        "note": "appliqué"})
    return serie, pd.DataFrame(journal)


serie, journal_periodes_niveau = ecarter_periodes(full_data["Niveau_(cm)"], PERIODES_ECARTEES_NIVEAU)
full_data["Niveau_(cm)"], journal_niveau = caler_sur_points_de_controle(
    serie, points_niveau, "Hauteur (cm)")

print("Périodes écartées :")
display(journal_periodes_niveau)
print("Décalages appliqués :")
display(journal_niveau)

graphe([(niveau_avant, "avant", "lightgrey"), (full_data["Niveau_(cm)"], "après", "green")],
       titre="Niveau — APRÈS correction", ylab="Niveau (cm)",
       points=points_niveau, col_point="Hauteur (cm)")

## 11. Conductivité - AVANT correction

In [ ]:
points_cond = pd.read_excel(PUNCTUAL_CONDUCT)
points_cond["Datetime"] = pd.to_datetime(points_cond["Jour"], dayfirst=True, errors="coerce")

conduct_avant = full_data["Conductivité"].copy()

graphe([(conduct_avant, "Conductivité (avant)", "grey")],
       titre="Conductivité — AVANT correction", ylab="Conductivité (µS/cm)",
       points=points_cond, col_point="Conductivité")

## 12. Conductivité — correction et graphe APRÈS

In [ ]:
# Périodes écartées au jugement : (début, fin, motif).
PERIODES_ECARTEES_COND = [
    ("2024-12-05 02:00", "2024-12-05 18:00", "pic anormal"),
    ("2023-06-20 13:00", "2023-09-05 12:00", "TROLL encrassée"),
    ("2019-06-27 10:00", "2019-10-23 18:00", "dérive CTD après nettoyage"),
    ("2019-02-27 15:00", "2019-04-05 12:00", "CTD hors d'eau"),
]

serie, journal_periodes_cond = ecarter_periodes(full_data["Conductivité"], PERIODES_ECARTEES_COND)
full_data["Conductivité"], journal_cond = caler_sur_points_de_controle(
    serie, points_cond, "Conductivité")

print("Périodes écartées :")
display(journal_periodes_cond)
print("Décalages appliqués :")
display(journal_cond)

graphe([(conduct_avant, "avant", "lightgrey"), (full_data["Conductivité"], "après", "green")],
       titre="Conductivité — APRÈS correction", ylab="Conductivité (µS/cm)",
       points=points_cond, col_point="Conductivité")

## 13. Filtre IQR sur la conductivité

Votre filtre, vectorisé : la boucle d'origine reconstruisait la fenêtre par comparaison de
dates sur tout le tableau à chaque pas, soit un coût quadratique (plusieurs minutes sur une
chronique pluriannuelle). Même résultat, quelques secondes.

In [ ]:
FENETRE_IQR, K_IQR = "48h", 0.8      # vos réglages d'origine

def filtre_iqr(serie, fenetre=FENETRE_IQR, k=K_IQR, min_periods=8):
    """Écarte ce qui sort de [Q1 − k·IQR, Q3 + k·IQR] sur fenêtre glissante centrée."""
    r = serie.rolling(fenetre, center=True, min_periods=min_periods)
    q1, q3 = r.quantile(0.25), r.quantile(0.75)
    iqr = q3 - q1
    return ((serie < q1 - k * iqr) | (serie > q3 + k * iqr)) & iqr.notna() & serie.notna()


ecartes_iqr = filtre_iqr(full_data["Conductivité"])
full_data["Conductivité_brute"] = full_data["Conductivité"]
full_data.loc[ecartes_iqr, "Conductivité"] = np.nan
full_data["Conductivité_Moyenne_Mobile"] = full_data["Conductivité"].rolling("6h", center=True).mean()

print(f"Filtre IQR ({FENETRE_IQR}, k={K_IQR}) : {int(ecartes_iqr.sum())} valeur(s) écartée(s) "
      f"({100 * ecartes_iqr.sum() / len(full_data):.2f} % de la chronique)")

graphe([(full_data["Conductivité_brute"], "avant IQR", "lightgrey"),
        (full_data["Conductivité"], "après IQR", "royalblue"),
        (full_data["Conductivité_Moyenne_Mobile"], "moyenne mobile 6 h", "green")],
       titre="Conductivité — filtre IQR", ylab="Conductivité (µS/cm)")

## 14. Débit

Courbe de tarage à deux branches raccordées à H = 26,1 cm, continue au seuil (0,0737 contre
0,0748). Le débit est calculé après les corrections du niveau, donc plus sur des valeurs
aberrantes.

In [ ]:
SEUIL_H = 26.1     # cm

def debit(H):
    H = pd.to_numeric(H, errors="coerce").astype("float64")
    Q = np.where(H > SEUIL_H, 0.0008 * H**2 - 0.0528 * H + 0.9079, 0.0027 * H + 0.0032)*1000
    return pd.Series(np.where(H.isna(), np.nan, Q), index=H.index)

full_data["Q_(L/s)"] = debit(full_data["Niveau_(cm)"])
full_data[["Niveau_(cm)", "Q_(L/s)"]].describe().round(3)

## 15. Interpolation des lacunes courtes

Les lacunes de moins de 12 h sont comblées, les plus longues restent des trous. La colonne
`Statut_<variable>` dit pour chaque pas si la valeur est **mesurée**, **interpolée** ou
**manquante**, et le graphe en fin de cellule le montre : ligne bleue pour les mesures,
points rouges pour les valeurs reconstruites, tirets gris en bas pour les lacunes restantes.
Changez la variable tracée sur la dernière ligne.

In [ ]:
MAX_TROU_H = 12

def interpoler_avec_statut(df, colonnes, max_trou_h=MAX_TROU_H, pas=PAS):
    """Comble les lacunes < max_trou_h et trace l'origine de chaque valeur."""
    df = df.copy()
    max_pas = int(pd.Timedelta(f"{max_trou_h}h") / pd.Timedelta(pas))
    for col in colonnes:
        origine = df[col]
        manquant = origine.isna().to_numpy()
        groupe = np.cumsum(np.r_[True, manquant[1:] != manquant[:-1]])
        tailles = pd.Series(groupe).groupby(groupe).transform("size").to_numpy()

        comble = origine.interpolate(method="time", limit_direction="both")
        comble[manquant & (tailles > max_pas)] = np.nan
        mesures = np.flatnonzero(~manquant)          # pas d'extrapolation hors plage mesurée
        if mesures.size:
            comble.iloc[:mesures[0]] = origine.iloc[:mesures[0]]
            comble.iloc[mesures[-1] + 1:] = origine.iloc[mesures[-1] + 1:]

        df[col] = comble
        df[f"Statut_{col}"] = np.where(~manquant, "Mesurée",
                               np.where(comble.notna().to_numpy(), "Interpolée", "Manquante"))
    return df


# Tous les paramètres de synthèse, donc automatiquement à jour si vous
# ajoutez une entrée à PRIORITES.
a_interpoler = [c for c in PRIORITES if c in full_data.columns]
full_data = interpoler_avec_statut(full_data, a_interpoler)
full_data["Q_(L/s)"] = debit(full_data["Niveau_(cm)"])   # recalculé sur le niveau interpolé
full_data["Statut_Q_(L/s)"] = full_data["Statut_Niveau_(cm)"]

display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in a_interpoler}).fillna(0).astype(int).T)


def graphe_interpolation(variable, sortie_html=None):
    """Montre ce qui est mesuré et ce qui a été reconstruit.

    Les valeurs interpolées sont isolées entre deux mesures : elles sont
    donc tracées en points, sur la courbe des valeurs mesurées.
    """
    statut = full_data[f"Statut_{variable}"]
    fig = go.Figure()
    fig.add_trace(go.Scattergl(x=full_data.index, y=full_data[variable].where(statut == "Mesurée"),
                               mode="lines", name="mesurée",
                               line=dict(color="royalblue", width=1.2)))
    interpolee = statut == "Interpolée"
    fig.add_trace(go.Scattergl(x=full_data.index[interpolee],
                               y=full_data.loc[interpolee, variable], mode="markers",
                               name=f"interpolée ({int(interpolee.sum())} pas)",
                               marker=dict(color="crimson", size=5)))
    manquante = statut == "Manquante"
    if manquante.any():
        fig.add_trace(go.Scattergl(x=full_data.index[manquante],
                                   y=np.full(int(manquante.sum()), full_data[variable].min()),
                                   mode="markers",
                                   name=f"manquante ({int(manquante.sum())} pas)",
                                   marker=dict(color="lightgrey", size=3, symbol="line-ns-open")))
    fig.update_layout(title=f"{variable} — origine de chaque valeur", xaxis_title="Date",
                      yaxis_title=variable, template="plotly_white", hovermode="x unified")
    if sortie_html:
        fig.write_html(sortie_html)
        print(f"Graphique sauvegardé : {sortie_html}")
    fig.show()


graphe_interpolation("Niveau_(cm)")      # ou "Conductivité", "Température"

## 16. Sauvegarde

Le fichier final ne contient **qu'un paramètre par grandeur** — c'est tout l'objet de la
fusion des trois sources — avec pour chacun son **statut** (mesurée / interpolée /
manquante) et la **voie** qui l'a fourni. Le détail capteur par capteur reste dans
`Fontbelle_consolide.xlsx`, écrit à la cellule 8.


In [ ]:
# Chronique finale : UN paramètre par grandeur, avec son statut et la voie
# qui l'a fournie. Le détail capteur par capteur reste dans SORTIE_CONSOLIDE.
PARAMETRES_FINAUX = list(PRIORITES) + ["Q_(L/s)"]

colonnes = []
for p in PARAMETRES_FINAUX:
    colonnes += [c for c in (p, f"Statut_{p}", f"{p}_source") if c in full_data.columns]
chronique = full_data[colonnes].copy()

onglets = {"periodes_ecartees_niveau": journal_periodes_niveau,
           "periodes_ecartees_cond": journal_periodes_cond,
           "calage_niveau": journal_niveau,
           "calage_conductivite": journal_cond}

with pd.ExcelWriter(SORTIE_FINALE) as writer:
    chronique.to_excel(writer, sheet_name="chronique")
    for nom, tableau in onglets.items():
        if tableau is not None and not tableau.empty:
            tableau.to_excel(writer, sheet_name=nom[:31], index=False)

print(f"{SORTIE_FINALE}")
print(f"  {len(chronique)} pas de temps × {chronique.shape[1]} colonnes")
print(f"  paramètres : {', '.join(PARAMETRES_FINAUX)}")
print(f"\n(détail capteur par capteur : {SORTIE_CONSOLIDE}, "
      f"{full_data.shape[1]} colonnes)")

display(pd.DataFrame({p: chronique[f"Statut_{p}"].value_counts()
                      for p in PARAMETRES_FINAUX
                      if f"Statut_{p}" in chronique.columns}).fillna(0).astype(int).T)
display(chronique.head())

## 17. Graphe de synthèse

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 10), sharex=True, gridspec_kw={"hspace": 0.05})

ax1 = axes[0]
ax1.plot(full_data.index, full_data["Q_(L/s)"].rolling(12, center=True).mean(), color="lightseagreen")
ax1.set_ylabel("Débit (L/s)", color="lightseagreen")
ax1.tick_params(axis="y", labelcolor="lightseagreen")

if os.path.exists(PLUIE_PATH):
    pluie = pd.read_csv(PLUIE_PATH)
    pluie["Date"] = pd.to_datetime(pluie["Date"], errors="coerce")
    pluie = pluie[(pluie["Date"] >= full_data.index.min()) & (pluie["Date"] <= full_data.index.max())]
    ax2 = ax1.twinx()
    ax2.bar(pluie["Date"], pluie["Precipitation (mm)"], width=0.8, color="royalblue")
    ax2.invert_yaxis()
    ax2.set_ylabel("Précipitations (mm)", color="royalblue")
    ax2.tick_params(axis="y", labelcolor="royalblue")

ax3 = axes[1]
ax3.plot(full_data.index, full_data["Conductivité_Moyenne_Mobile"], color="black")
ax3.set_ylabel("Conductivité (µS/cm)", color="black")
ax4 = ax3.twinx()
ax4.plot(full_data.index, full_data["Température"].rolling(24, center=True).mean(), color="crimson")
ax4.set_ylabel("Température (°C)", color="crimson")
ax4.tick_params(axis="y", labelcolor="crimson")

ax5 = axes[2]
for col, couleur, label, axe, ecart in [
        ("Turbidité_(NTU)", "darkorange", "Turbidité (NTU)", ax5, 0),
        ("O2_(mg/l)", "darkmagenta", "Oxygène (mg/L)", ax5.twinx(), 0),
        ("Chlorophylle_(RFU)", "green", "Chlorophylle (RFU)", ax5.twinx(), 45)]:
    if col not in full_data.columns:
        continue
    if ecart:
        axe.spines["right"].set_position(("outward", ecart))
    axe.plot(full_data.index, full_data[col].rolling(24, center=True).mean(), color=couleur)
    axe.set_ylabel(label, color=couleur)
    axe.tick_params(axis="y", labelcolor=couleur)

plt.savefig(SORTIE_SVG, format="svg", dpi=300, bbox_inches="tight")
print(f"Graphique sauvegardé : {SORTIE_SVG}")
plt.show()

## 18. Comparaison des sources

À lancer seulement quand vous voulez vérifier une bascule de capteur. Les séries sont
moyennées par jour avant tracé : quelques centaines de points au lieu de plusieurs dizaines
de milliers, donc rien qui rame.

In [ ]:
PARAMETRE = "Conductivité"      # ou "Niveau_(cm)", "Température"

jour = full_data.resample("1D").mean(numeric_only=True)
traces = [(jour[c], c, coul) for c, coul in
          zip(PRIORITES[PARAMETRE], ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"])
          if c in jour.columns and jour[c].notna().any()]
traces.append((jour[PARAMETRE], f"{PARAMETRE} (retenu)", "black"))

graphe(traces, titre=f"{PARAMETRE} — voies candidates (moyennes journalières)", ylab=PARAMETRE)

print("Répartition des pas de temps par voie retenue :")
display(full_data[f"{PARAMETRE}_source"].value_counts())